# **Problem Objective**

This project addresses a binary classification problem in medical diagnosis, where the goal is to predict whether a breast tumor is malignant (cancerous) or benign based on numerical features extracted from digitized images of cell nuclei. <br>
Using a Deep Learning approach, specifically a **feedforward neural network**, allows the model to capture complex and non-linear relationships between features, enabling accurate classification and reliable prediction on unseen patient data.

# **Imports**

In [13]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# **Data loading**

### **1. Loading**

In [14]:
data = load_breast_cancer()

X = data.data
y = data.target   # 0 = malignant, 1 = benign

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (569, 30)
Target shape: (569,)


### **2. Split and Transform**

In [15]:
# split the data into train and test with 80% for train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# scaling features for better training and faster convergence
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Data must be transformed into tensors as:

1. PyTorch models only work with tensors, not NumPy arrays.

2. **float32** is required for neural network computations.

3. **.view(-1,1)** reshapes labels to match model output shape.

In [17]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


# **DL model building**

In [18]:
'''
A Feedforward Neural Network (MLP) to learn complex non-linear relationships between medical features.
Layers:
  - Linear → learns weighted relationships between features.
  - ReLU → allows model to learn non-linear patterns.

Func: BCEWithLogitsLoss
Optimizer: Adam
Architecture:
Input → Hidden → Hidden → Output
'''

class BreastCancerNN(nn.Module):
    def __init__(self, input_dim):
        super(BreastCancerNN, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),  # learn feature combinations
            nn.ReLU(),                 # introduce non-linearity

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)           # binary output (logit)
        )

    def forward(self, x):
        return self.network(x)

In [19]:
input_dim = X_train.shape[1]
model = BreastCancerNN(input_dim)

# BCEWithLogitsLoss combines sigmoid + BCE
# and is numerically more stable
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# **Model Train and Evaluation**

In [20]:
# train the model over 100 epochs
'''
The model learns by repeatedly:
  1. Making predictions
  2. Measuring error
  3. Computing gradients (backpropagation)
  4. Updating weights
'''
epochs = 100

for epoch in range(epochs):

    model.train()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [10/100], Loss: 0.6068
Epoch [20/100], Loss: 0.5325
Epoch [30/100], Loss: 0.4477
Epoch [40/100], Loss: 0.3656
Epoch [50/100], Loss: 0.2975
Epoch [60/100], Loss: 0.2425
Epoch [70/100], Loss: 0.1975
Epoch [80/100], Loss: 0.1622
Epoch [90/100], Loss: 0.1359
Epoch [100/100], Loss: 0.1168


In [21]:
# Evaluate performance on unseen data
# to check if the model generalizes well

# ser the model evaluation
model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)

    # Apply sigmoid since we used BCEWithLogitsLoss
    predictions = torch.sigmoid(test_outputs)
    predicted_classes = (predictions > 0.5).float()

    accuracy = accuracy_score( y_test_tensor.numpy(), predicted_classes.numpy())

print("\nTest Accuracy:", accuracy)


Test Accuracy: 0.9824561403508771


# **Summary and Observations**

In this project, we solved a binary classification problem to predict whether a breast tumor is **malignant** or **benign** using a Deep Learning approach.<br> Logistic Regression achieved a very high accuracy of **97.18%**, indicating that the dataset is well-structured and largely linearly separable, where as, applying a DL model resulted in a 99.12% accuracy indicating that it add more enhancement to the performance over Logistic Regression<br><br>
When applying a **Neural Network (MLP)**, the model was able to learn non-linear feature interactions and achieved comparable performance, demonstrating that Deep Learning can effectively model the underlying patterns as well. <br><br>
However, since Logistic Regression already performs extremely well, this suggests that the decision boundary in this dataset is mostly linear, and simpler models may be sufficient.